# HERMES five-stage AutoDL research suite — fresh clone

Run **Kernel → Restart & Run All** after cloning the repository. This notebook completes five new stages without rerunning the preserved 201–250 original/repeat/FIFO experiments:

1. real-video pseudo-query attention validation;
2. non-overlapping samples 251–300 generalization (HERMES/FIFO at 4K/6K, plus 50K where feasible);
3. four isolated component ablations at 4K on 251–300;
4. three same-session interleaved HERMES/FIFO 4K timing repetitions on ten videos;
5. resumable all-498-video evaluation for the predeclared primary HERMES-4K versus FIFO-4K comparison, reusing 201–250 and 251–300.

The full benchmark needs roughly 120 GB of downloads in total, but archives/videos are processed and cleaned one shard at a time. Keep at least **55 GiB free** and use a GPU with at least **15 GiB VRAM**. The 50K generalization control is allowed to finish as an explicit feasibility failure if the GPU cannot hold an untruncated cache; no input truncation or silent omission is permitted.

All original predictions/logs are immutable inputs. New inference is checkpointed per video. Partial evidence is rejected, official tracked source is unchanged, and synthetic self-tests never enter research results. The final ZIP contains preserved evidence, raw predictions, logs, scripts, timing events, analyses, figures, provenance, report, and a SHA-256 manifest.

In [ ]:
!source /etc/network_turbo

In [ ]:
# Bootstrap notebook-kernel packages before importing them.
import sys, subprocess, importlib.util
_requirements = {'numpy':'numpy','pandas':'pandas','scipy':'scipy','matplotlib':'matplotlib','requests':'requests'}
_missing = [package for module, package in _requirements.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-i',
                           'https://pypi.tuna.tsinghua.edu.cn/simple',*_missing])
from pathlib import Path
import os, sys, subprocess, json, re, zipfile, hashlib, shutil, time, copy, platform, gc, ast
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.stats import binomtest
import matplotlib.pyplot as plt
from IPython.display import display
import requests

MODEL_ID='llava-hf/llava-onevision-qwen2-0.5b-ov-hf'
MODEL_SHA='74dd0bf867a4cda7950c17663794267c60cf4b40'
HERMES_SHA='8d699b16a6bedb9086c1b39ec4253c6a1d1ce789'
TF_SHA='66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
FPS,SEED=0.5,2024
BOOTSTRAP_REPS=10000
ROOT=Path(os.environ.get('HERMES_REPO','/root/autodl-tmp/HERMES-reproduction')).expanduser().resolve()
if not ROOT.is_dir():
    candidates=[Path('/root/autodl-tmp/HERMES-reproduction'),Path('/root/autodl-tmp/HERMES'),Path.cwd()]
    ROOT=next((x.resolve() for x in candidates if (x/'inference/llavaov_hermes.py').is_file()),ROOT)
WORK=Path(os.environ.get('HERMES_WORK','/root/autodl-tmp/hermes_five_stage')).expanduser().resolve()
PRESERVED=Path(os.environ.get('HERMES_PRIOR_EVIDENCE',str(ROOT/'HERMES_AutoDL_final_evidence'))).resolve()
assert ROOT.is_dir() and WORK!=ROOT and ROOT not in WORK.parents and WORK not in ROOT.parents
for rel in ('inference/llavaov_hermes.py','video_qa/hermes_vqa.py',
            'data/streamingbench/streamingbench_realtime.json'):
    assert (ROOT/rel).is_file(),ROOT/rel
assert PRESERVED.is_dir(),f'Unpacked prior evidence is required: {PRESERVED}'
RESULTS,DATA,EXPORTS,LOGS=WORK/'results',WORK/'data',WORK/'exports',WORK/'logs'
FIGURES=EXPORTS/'figures'
VENV=WORK/'venvs'/'hermes-py312';PY312=VENV/'bin'/'python'
ARCHIVE_CACHE=WORK/'downloads';HF_HOME=WORK/'cache'/'huggingface'
MODEL_CACHE=HF_HOME/'hub';UV_CACHE=WORK/'cache'/'uv'
EXPECTED_MODEL_SNAPSHOT=MODEL_CACHE/'models--llava-hf--llava-onevision-qwen2-0.5b-ov-hf'/'snapshots'/MODEL_SHA
for directory in (WORK,RESULTS,DATA,EXPORTS,FIGURES,LOGS,ARCHIVE_CACHE,HF_HOME,MODEL_CACHE,UV_CACHE):
    directory.mkdir(parents=True,exist_ok=True)

def run(args,*,cwd=None,env=None,check=True):
    proc=subprocess.run([str(a) for a in args],cwd=cwd,env=env,text=True,capture_output=True)
    if check and proc.returncode:
        raise RuntimeError('Command failed: '+repr(args)+'\n'+(proc.stdout+proc.stderr)[-6000:])
    return proc

def run_live(args,*,cwd=None,env=None):
    proc=subprocess.run([str(a) for a in args],cwd=cwd,env=env)
    if proc.returncode: raise RuntimeError(f'Command failed ({proc.returncode}): {args!r}')
    return proc

def run_logged(args,log_path,*,env=None,cwd=ROOT):
    log_path=Path(log_path);log_path.parent.mkdir(parents=True,exist_ok=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND_REDACTED_TO_ARGS_ONLY '+json.dumps([str(a) for a in args])+'\n');log.flush()
        proc=subprocess.Popen([str(a) for a in args],cwd=cwd,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line)
        code=proc.wait()
    if code: raise RuntimeError(f'Process failed with code {code}; complete log: {log_path}')

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def download_archive(urls,dest,retries=4):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True);errors=[]
    if isinstance(urls,str):urls=[urls]
    for url in urls:
        for attempt in range(retries):
            try:
                headers={};mode='wb'
                if dest.exists() and dest.stat().st_size:
                    headers['Range']=f'bytes={dest.stat().st_size}-';mode='ab'
                with requests.Session() as session:
                    for key in ('HTTP_PROXY','HTTPS_PROXY','ALL_PROXY','NO_PROXY','http_proxy','https_proxy','all_proxy','no_proxy'):
                        if key in NETWORK_ENV:session.proxies[key]=NETWORK_ENV[key]
                    with session.get(url,headers=headers,stream=True,timeout=(30,120)) as response:
                        if response.status_code==416 and dest.exists():return dest
                        if response.status_code!=206:mode='wb'
                        response.raise_for_status()
                        with open(dest,mode) as handle:
                            for block in response.iter_content(1024*1024):
                                if block:handle.write(block)
                assert dest.stat().st_size>0;return dest
            except Exception as exc:
                errors.append(f'{url} attempt {attempt+1}: {exc}');print(errors[-1],flush=True);time.sleep(5*(attempt+1))
    raise RuntimeError('Archive download failed:\n'+'\n'.join(errors[-6:]))
def network_environment():
    env=os.environ.copy();turbo=Path('/etc/network_turbo')
    if turbo.is_file():
        proc=subprocess.run(['bash','-lc','source /etc/network_turbo >/dev/null 2>&1; env -0'],capture_output=True)
        if proc.returncode==0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key,value=item.split(b'=',1);env[key.decode()]=value.decode()
    return env
NETWORK_ENV=network_environment()
assert Path('/root/autodl-tmp').is_dir()
free=shutil.disk_usage(WORK).free/1024**3
assert free>=55,f'Need at least 55 GiB free for one large shard; found {free:.1f}'
smi=run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=False)
assert smi.returncode==0 and smi.stdout.strip(),smi.stderr
print('GPU:',smi.stdout.strip())
mem=float(re.search(r'(\d+(?:\.\d+)?)\s*MiB',smi.stdout).group(1))/1024
assert mem>=15,f'At least 15 GiB VRAM required; found {mem:.1f}'
print('Repository:',ROOT,'\nWork:',WORK,'\nPreserved evidence:',PRESERVED,'\nFree GiB:',round(free,1))
print('Completed 201–250 inference is DISABLED; only validated preserved predictions are reused.')

## 1. Pinned environment, source identity, and model snapshot

Installation is skipped when the exact environment probe passes. PyTorch uses the direct CPython 3.12 CUDA 12.8 wheel because index resolution has failed on fresh AutoDL containers.

In [ ]:
_env_probe=r"""
import sys,json,torch,transformers,tokenizers,huggingface_hub
import importlib.metadata as md
assert sys.version_info[:2]==(3,12)
assert torch.__version__=='2.10.0+cu128'
assert transformers.__version__=='4.45.0.dev0'
assert tokenizers.__version__=='0.19.1' and huggingface_hub.__version__=='0.34.4'
meta=json.loads(md.distribution('transformers').read_text('direct_url.json'))
if 'vcs_info' in meta: assert meta['vcs_info']['commit_id']=='66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
else: assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in meta.get('url','')
import decord,logzero,pandas,scipy,av
"""
_env_ok=PY312.is_file() and run([PY312,'-c',_env_probe],check=False).returncode==0
if _env_ok:
    print('Reusing validated pinned environment:',PY312)
else:
    try:run([sys.executable,'-m','uv','--version'])
    except RuntimeError:run([sys.executable,'-m','pip','install','-q','-i','https://pypi.tuna.tsinghua.edu.cn/simple','uv'])
    UV=[sys.executable,'-m','uv']
    INSTALL_ENV={**NETWORK_ENV,'UV_CACHE_DIR':str(UV_CACHE),'UV_LINK_MODE':'copy','TMPDIR':str(WORK),
                 'UV_DEFAULT_INDEX':'https://pypi.tuna.tsinghua.edu.cn/simple'}
    if not PY312.exists():run(UV+['venv',str(VENV),'--python','3.12','--seed'],env=INSTALL_ENV)
    def uv_install(packages,extra=()):
        proc=run(UV+['pip','install','--python',str(PY312)]+list(extra)+list(packages),env=INSTALL_ENV)
        if proc.stderr:print(proc.stderr[-1800:])
    uv_install(['torch @ https://download-r2.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl'])
    uv_install(['numpy==1.26.4','tokenizers==0.19.1','huggingface-hub==0.34.4','accelerate==0.34.2',
        'decord==0.6.0','logzero','pillow','sentencepiece','safetensors','packaging','filelock','regex',
        'requests','pyyaml','tqdm','fsspec','protobuf','einops','psutil','pandas','scipy','av'])
    tf_archive=ARCHIVE_CACHE/f'transformers-{TF_SHA}.zip'
    if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
        tf_archive.unlink(missing_ok=True)
        url=f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}'
        download_archive([f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}',
                          f'https://github.com/huggingface/transformers/archive/{TF_SHA}.zip'],tf_archive)
    assert zipfile.is_zipfile(tf_archive)
    uv_install([str(tf_archive)],['--no-deps'])
check_code=r"""
import sys,json,inspect,torch,transformers,tokenizers,huggingface_hub
import importlib.metadata as md
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
assert sys.version_info[:2]==(3,12) and torch.cuda.is_available()
assert torch.__version__=='2.10.0+cu128' and transformers.__version__=='4.45.0.dev0'
assert tokenizers.__version__=='0.19.1' and huggingface_hub.__version__=='0.34.4'
assert 'seq_len' in str(inspect.signature(Qwen2RotaryEmbedding.forward))
print('Torch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0))
print('Transformers:',transformers.__version__,'pinned environment: PASS')
"""
model_env={**NETWORK_ENV,'HF_HOME':str(HF_HOME),'HF_HUB_DISABLE_XET':'1','HF_ENDPOINT':'https://hf-mirror.com',
           'HF_HUB_ENABLE_HF_TRANSFER':'0','CUDA_VISIBLE_DEVICES':'0'}
print(run([PY312,'-c',check_code],env=model_env).stdout)
SUBENV={**model_env,'PYTHONPATH':str(ROOT)+os.pathsep+os.environ.get('PYTHONPATH',''),
        'TOKENIZERS_PARALLELISM':'false','PYTORCH_ALLOC_CONF':'expandable_segments:True'}

In [ ]:
# The reproduction repository has its own Git HEAD; do not confuse it with official HERMES_SHA.
assert (ROOT/'.git').exists(), f'Expected a Git clone at {ROOT}'
REPRO_HEAD = run(['git','rev-parse','HEAD'], cwd=ROOT).stdout.strip()
SOURCE_PATHS = ['inference','video_qa','data/streamingbench/streamingbench_realtime.json']
source_commit = run(['git','cat-file','-e',HERMES_SHA+'^{commit}'], cwd=ROOT, check=False)
if source_commit.returncode == 0:
    diff = run(['git','diff','--name-only',HERMES_SHA,'--']+SOURCE_PATHS, cwd=ROOT).stdout.splitlines()
    assert not diff, f'Cloned source differs from pinned official HERMES: {diff}'
    untracked = run(['git','ls-files','--others','--exclude-standard','--','inference','video_qa'], cwd=ROOT).stdout.splitlines()
    unexpected = [x for x in untracked if not x.endswith('.pyc') and '__pycache__/' not in x]
    assert not unexpected, f'Unexpected untracked inference source: {unexpected}'
    SOURCE_VERIFICATION = f'git diff against official commit {HERMES_SHA}'
else:
    # Shallow reproduction clones may omit the official commit; compare a pinned archive instead.
    archive = ARCHIVE_CACHE/f'HERMES-{HERMES_SHA}.zip'
    if not archive.is_file() or not zipfile.is_zipfile(archive):
        archive.unlink(missing_ok=True)
        download_archive([f'https://codeload.github.com/haowei-freesky/HERMES/zip/{HERMES_SHA}',
                          f'https://github.com/haowei-freesky/HERMES/archive/{HERMES_SHA}.zip'], archive)
    assert zipfile.is_zipfile(archive), archive
    official_extract = WORK/'source_verify'/HERMES_SHA
    if official_extract.exists(): shutil.rmtree(official_extract)
    official_extract.mkdir(parents=True)
    with zipfile.ZipFile(archive) as z:
        base = official_extract.resolve()
        for item in z.infolist():
            target = (official_extract/item.filename).resolve()
            assert target == base or base in target.parents, item.filename
        z.extractall(official_extract)
    official_root = next(x for x in official_extract.iterdir() if x.is_dir())
    for area in ('inference','video_qa'):
        def source_files(base):
            return {str(x.relative_to(base)) for x in (base/area).rglob('*')
                    if x.is_file() and not x.name.endswith('.pyc') and '__pycache__' not in x.parts}
        assert source_files(ROOT) == source_files(official_root), f'Source file set differs in {area}'
        for rel in source_files(official_root):
            assert sha256(ROOT/rel) == sha256(official_root/rel), rel
    annotation_rel = 'data/streamingbench/streamingbench_realtime.json'
    assert sha256(ROOT/annotation_rel) == sha256(official_root/annotation_rel)
    SOURCE_VERIFICATION = f'byte comparison with official pinned archive {HERMES_SHA}'
for rel in ('inference/llavaov_hermes.py','inference/reindex_1d.py','video_qa/base.py',
            'data/streamingbench/streamingbench_realtime.json'):
    assert (ROOT/rel).is_file(), rel
    print(rel, sha256(ROOT/rel))
reindex_tail = (ROOT/'inference/reindex_1d.py').read_text().split('def apply_rotary_delta_to_keys_only')[-1][:230]
assert 'return key_states' not in reindex_tail, 'Disabled RoPE-delta debug patch detected.'
source_provenance = {'reproduction_repository':str(ROOT), 'reproduction_head':REPRO_HEAD,
                     'official_hermes_revision':HERMES_SHA, 'verification':SOURCE_VERIFICATION}
(WORK/'source_provenance.json').write_text(json.dumps(source_provenance,indent=2))
print('Reproduction HEAD:', REPRO_HEAD)
print('Official HERMES source fidelity: PASS |', SOURCE_VERIFICATION)

In [ ]:
def valid_model_directory(path):
    path=Path(path)
    return path.is_dir() and (path/'config.json').is_file() and bool(list(path.glob('*.safetensors')))
if valid_model_directory(EXPECTED_MODEL_SNAPSHOT):
    MODEL_SNAPSHOT=EXPECTED_MODEL_SNAPSHOT.resolve();print('Reusing pinned model cache:',MODEL_SNAPSHOT)
else:
    downloader=WORK/'download_model.py';pointer=WORK/'model_snapshot_path.json'
    downloader.write_text(r"""
import json,sys
from huggingface_hub import snapshot_download
path=snapshot_download(repo_id=sys.argv[1],revision=sys.argv[2],cache_dir=sys.argv[3],
 allow_patterns=['*.json','*.safetensors','*.txt','*.model','*.vocab','tokenizer.*','merges.txt'],
 ignore_patterns=['onnx/*','*.onnx','*.bin','*.msgpack'])
with open(sys.argv[4],'w') as f:json.dump(str(path),f)
print('MODEL_SNAPSHOT='+str(path),flush=True)
""")
    pointer.unlink(missing_ok=True)
    run_live([PY312,str(downloader),MODEL_ID,MODEL_SHA,str(MODEL_CACHE),str(pointer)],env=model_env)
    MODEL_SNAPSHOT=Path(json.loads(pointer.read_text())).resolve()
assert valid_model_directory(MODEL_SNAPSHOT) and MODEL_SNAPSHOT.name==MODEL_SHA
print('Model snapshot identity: PASS |',MODEL_SNAPSHOT)

## 2. Verify all preserved evidence and establish strict CSV identity

Every payload in the prior archive’s unpacked manifest is hash-checked. All seven prior prediction CSVs are re-read and aligned by `(video_id, qa_index)`. None is regenerated.

In [ ]:
manifest=pd.read_csv(PRESERVED/'exports/final/master_manifest_sha256.csv')
assert len(manifest)==53 and manifest.path.is_unique
for row in manifest.itertuples():
    path=PRESERVED/row.path
    assert path.is_file() and path.stat().st_size==row.bytes and sha256(path)==row.sha256,row.path
actual={p.relative_to(PRESERVED).as_posix() for p in PRESERVED.rglob('*') if p.is_file()}
assert actual==set(manifest.path)|{'exports/final/master_manifest_sha256.csv'}
print('PRIOR EVIDENCE: 53/53 payload hashes PASS; no unmanifested payload')

ANNOTATIONS=json.loads((ROOT/'data/streamingbench/streamingbench_realtime.json').read_text())
assert len(ANNOTATIONS)==498 and sum(len(x['conversations']) for x in ANNOTATIONS)==2495
KEY=['video_id','qa_index']
def annotation_sample_id(item):
    match=re.search(r'sample_(\d+)_real\.mp4',str(item['video_path']))
    assert match,item['video_path']
    return int(match.group(1))
def annotation_identity(records):
    return [{'video_id':str(x['video_id']),'conversations':x['conversations']} for x in records]
def read_predictions(path,records,label=None,verbose=True):
    path=Path(path);d=pd.read_csv(path).copy();lookup={str(x['video_id']):x for x in records}
    expected=sum(len(x['conversations']) for x in records)
    assert len(d)==expected and d.video_id.nunique()==len(records),(path,len(d),expected)
    d['video_id']=d.video_id.astype(str);d['qa_index']=d.groupby('video_id',sort=False).cumcount().astype(int)
    assert not d.duplicated(KEY).any() and not d[['question','qa_acc']].isna().any().any()
    assert pd.to_numeric(d.qa_acc,errors='coerce').isin([0,100]).all()
    for row in d.itertuples():
        qa=lookup[row.video_id]['conversations'][row.qa_index]
        assert str(row.question).strip()==str(qa['question']).strip()
        assert ast.literal_eval(row.choices)==qa['choices']
        letter='ABCDEFGH'[qa['choices'].index(qa['answer'])]
        assert str(row.correct_choice)==letter
        assert (str(row.pred_choice).strip().upper()==letter)==(float(row.qa_acc)==100.0)
    d['correct']=pd.to_numeric(d.qa_acc).eq(100).astype(int)
    if verbose:print('CSV PASS:',label or path,len(d),'rows',d.video_id.nunique(),'videos')
    return d
PRIOR50=json.loads((PRESERVED/'data/streamingbench_50_autodl.json').read_text())
prior_paths={
 'HERMES_4000':'preserved/HERMES/results/streamingbench_50_kv4000/1_0.csv',
 'HERMES_6000':'preserved/HERMES/results/streamingbench_50_kv6000/1_0.csv',
 'HERMES_50000':'preserved/HERMES/results/streamingbench_50_kv50000/1_0.csv',
 'HERMES_4000_REPEAT1':'results/streamingbench_50_kv4000_autodl_repeat1/1_0.csv',
 'HERMES_6000_REPEAT1':'results/streamingbench_50_kv6000_autodl_repeat1/1_0.csv',
 'FIFO_4000':'results/streamingbench_50_kv4000_fifo/1_0.csv',
 'FIFO_6000':'results/streamingbench_50_kv6000_fifo/1_0.csv'}
PRIOR={name:read_predictions(PRESERVED/path,PRIOR50,name) for name,path in prior_paths.items()}
print('All seven preserved generations validated; no inference launched.')

## 3. StreamingBench shard assets

The ten archive names, byte sizes, and LFS SHA-256 values below come from the dataset file listing. Downloads are directly resumable and hash-verified. A complete directory marker records every extracted video’s hash; stray macOS `._*` files are excluded.

In [ ]:
SHARDS=[
 (1,50,'Real-Time Visual Understanding_1-50.zip',9080551796,'39f8e42130424bddfa8c298be882b21fa3e818318e9782e28ef705851c0c82c5'),
 (51,100,'Real-Time Visual Understanding_51-100.zip',8708188924,'6a2b3317008e2f2c4cb30046bf74fedb5c8a75e18b6a042a0949c6032224a934'),
 (101,150,'Real-Time Visual Understanding_101-150.zip',8218047488,'6fdc44e97d1888b471b2d7f55dddace16291c04fdfcaab10433b35fb315bb8b3'),
 (151,200,'Real-Time Visual Understanding_151-200.zip',17984397733,'ce4c47e4161a5502f25e9fe9b7ee3929aa839b6916777e3614a81fa31def4472'),
 (201,250,'Real-Time Visual Understanding_201-250.zip',2230277873,'b7946e8b678fbf704b9816fa708e6febb77f5c37a7ad884084adf858f5d20dbf'),
 (251,300,'Real-Time Visual Understanding_251-300.zip',18838513104,'776f78a9338aaa72ed2e8c99a10eeced813c8499988bda29f4205c542a8a577a'),
 (301,350,'Real-Time Visual Understanding_301-350.zip',16909427777,'d627530553789f0dc4c948fb39ce1cff3c88a9fa47aa8e385da54566f5dde10e'),
 (351,400,'Real-Time Visual Understanding_351-400.zip',17902292310,'d46eb244909af5a06cf5243d691a8544cd000ea2d096ef5a7c45e951cb94aa15'),
 (401,450,'Real-Time Visual Understanding_401-450.zip',18224908176,'71e011d82b559534adc877d78d11e570b20a7fce015963ce35753d6b923dfa52'),
 (451,500,'Real-Time Visual Understanding_451-500.zip',13468580983,'63a6df5db049b3fd841e37bd8772e59e90654e1bd5214cba0332745a3feb5477')]
SHARD_MAP={(a,b):{'filename':n,'bytes':size,'sha256':sha} for a,b,n,size,sha in SHARDS}
assert sorted(set(range(a,b+1) for a,b, *_ in SHARDS),key=lambda x:x[0])
def safe_extract(zip_path,output):
    output=Path(output);base=output.resolve();total=0
    with zipfile.ZipFile(zip_path) as archive:
        for item in archive.infolist():
            target=(output/item.filename).resolve()
            assert target==base or base in target.parents,item.filename
            assert not item.is_dir() or not item.filename.startswith('/')
            total+=item.file_size
        assert total<=60*1024**3,total/1024**3
        archive.extractall(output)
def annotation_for_directory(path,start,end):
    candidates={}
    for obj in Path(path).rglob('*'):
        if obj.is_file() and obj.suffix.lower() in {'.mp4','.avi','.mov','.mkv'} and not obj.name.startswith('._'):
            matches=re.findall(r'sample_(\d+)',str(obj))
            if matches and start<=int(matches[-1])<=end:
                sample=int(matches[-1]);assert sample not in candidates,(sample,obj)
                candidates[sample]=obj.resolve()
    mapped=[]
    for item in ANNOTATIONS:
        sample=annotation_sample_id(item)
        if start<=sample<=end:
            assert sample in candidates,f'Missing sample_{sample} in {path}'
            record=copy.deepcopy(item);record['video_path']=str(candidates[sample]);mapped.append((sample,record))
    return [record for _,record in sorted(mapped)]
def download_file(urls,dest,expected_bytes,expected_sha):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.is_file() and dest.stat().st_size==expected_bytes and sha256(dest)==expected_sha:
        print('Reusing hash-verified archive:',dest);return dest
    errors=[]
    for url in urls:
        for attempt in range(4):
            try:
                headers={};mode='wb'
                if dest.exists() and dest.stat().st_size<expected_bytes:
                    headers['Range']=f'bytes={dest.stat().st_size}-';mode='ab'
                with requests.Session() as session:
                    for key in ('HTTP_PROXY','HTTPS_PROXY','ALL_PROXY','NO_PROXY','http_proxy','https_proxy','all_proxy','no_proxy'):
                        if key in NETWORK_ENV:session.proxies[key]=NETWORK_ENV[key]
                    with session.get(url,headers=headers,stream=True,timeout=(30,180)) as response:
                        if response.status_code==416 and dest.stat().st_size==expected_bytes:break
                        if response.status_code!=206:mode='wb'
                        response.raise_for_status();start=time.time();last=start
                        with open(dest,mode) as handle:
                            for block in response.iter_content(4*1024*1024):
                                if block:handle.write(block)
                                if time.time()-last>30:
                                    print(f'{dest.name}: {dest.stat().st_size/1024**3:.2f}/{expected_bytes/1024**3:.2f} GiB',flush=True);last=time.time()
                assert dest.stat().st_size==expected_bytes,(dest.stat().st_size,expected_bytes)
                assert sha256(dest)==expected_sha,'Archive SHA-256 mismatch'
                return dest
            except Exception as exc:
                errors.append(f'{url} attempt {attempt+1}: {exc}');print(errors[-1],flush=True);time.sleep(5*(attempt+1))
    raise RuntimeError('Download failed:\n'+'\n'.join(errors[-6:]))
def video_manifest(records):
    return [{'name':Path(x['video_path']).name,'bytes':Path(x['video_path']).stat().st_size,
             'sha256':sha256(x['video_path'])} for x in records]
def prepare_shard(start,end):
    spec=SHARD_MAP[(start,end)];video_dir=WORK/'videos'/f'streamingbench_{start}_{end}'
    marker=video_dir/'dataset_complete.json';annotation_path=DATA/f'streamingbench_{start}_{end}.json'
    if marker.is_file() and annotation_path.is_file():
        records=json.loads(annotation_path.read_text());saved=json.loads(marker.read_text())
        assert len(records)==50 and saved['status']=='complete'
        for item,meta in zip(records,saved['videos']):
            path=Path(item['video_path']);assert path.is_file() and path.stat().st_size==meta['bytes']
            assert sha256(path)==meta['sha256'],path
        print('Reusing complete hash-verified videos:',video_dir);return records,video_dir,annotation_path
    # Reuse an existing complete old-shard directory, but hash every video before accepting it.
    if start==201 and not video_dir.exists():
        for candidate in (ROOT/'HERMES/videos/streamingbench_201_250',ROOT/'data/streamingbench/videos'):
            if candidate.is_dir() and len(annotation_for_directory(candidate,201,250))==50:
                video_dir=candidate.resolve();break
    if not video_dir.exists() or len(annotation_for_directory(video_dir,start,end))!=50:
        assert shutil.disk_usage(WORK).free/1024**3>=55,'Insufficient free space for archive + extraction'
        archive=ARCHIVE_CACHE/spec['filename'];quoted=requests.utils.quote(spec['filename'])
        urls=[f'https://hf-mirror.com/datasets/mjuicem/StreamingBench/resolve/main/{quoted}?download=true',
              f'https://huggingface.co/datasets/mjuicem/StreamingBench/resolve/main/{quoted}?download=true']
        download_file(urls,archive,spec['bytes'],spec['sha256'])
        video_dir.mkdir(parents=True,exist_ok=True);safe_extract(archive,video_dir)
    records=annotation_for_directory(video_dir,start,end)
    assert len(records)==50 and sum(len(x['conversations']) for x in records)==250
    annotation_path.write_text(json.dumps(records,indent=2))
    marker_data={'status':'complete','shard':[start,end],'archive_filename':spec['filename'],
                 'archive_sha256':spec['sha256'],'videos':video_manifest(records)}
    (video_dir/'dataset_complete.json').write_text(json.dumps(marker_data,indent=2))
    (DATA/f'streamingbench_{start}_{end}_assets.json').write_text(json.dumps(marker_data,indent=2))
    archive=ARCHIVE_CACHE/spec['filename'];archive.unlink(missing_ok=True)
    print(f'SHARD READY {start}-{end}: 50 videos, 250 questions; archive removed after verification')
    return records,video_dir,annotation_path

In [ ]:
SCRIPTS=WORK/'scripts';SCRIPTS.mkdir(exist_ok=True)
RUNNER=SCRIPTS/'experiment_runner.py'
RUNNER.write_text(r'''#!/usr/bin/env python3
"""Resumable HERMES/FIFO/ablation runner. Official tracked source is never edited."""
import argparse
import ast
import contextlib
import csv
import gc
import hashlib
import inspect
import json
import os
import random
import re
import sys
import tempfile
import textwrap
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch

KEY = ["video_id", "qa_index"]


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def stable_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def annotation_identity(records):
    return [{"video_id": str(x["video_id"]), "conversations": x["conversations"]} for x in records]


def safe_name(value):
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))
    assert value and value not in {".", ".."}
    return value


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False))
    os.replace(temporary, path)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, quoting=csv.QUOTE_NONNUMERIC)
    os.replace(temporary, path)


def expected_rows(video):
    rows = []
    for index, qa in enumerate(video["conversations"]):
        choices = qa.get("choices")
        answer = qa.get("answer")
        assert choices and answer in choices and choices.count(answer) == 1
        rows.append({"question": str(qa["question"]), "choices": choices,
                     "answer": answer, "correct_choice": "ABCDEFGH"[choices.index(answer)],
                     "task": qa.get("task", qa.get("question_type", video.get("task")))})
    return rows


def validate_video_csv(path, video):
    frame = pd.read_csv(path)
    expected = expected_rows(video)
    assert len(frame) == len(expected), (path, len(frame), len(expected))
    assert frame.video_id.astype(str).eq(str(video["video_id"])).all()
    assert not frame[["question", "pred_answer", "pred_choice", "qa_acc"]].isna().any().any()
    assert pd.to_numeric(frame.qa_acc, errors="coerce").isin([0, 100]).all()
    for index, row in frame.iterrows():
        target = expected[index]
        assert str(row.question).strip() == target["question"].strip()
        assert ast.literal_eval(row.choices) == target["choices"]
        assert str(row.answer) == str(target["answer"])
        assert str(row.correct_choice) == target["correct_choice"]
        parsed = str(row.pred_choice).strip().upper() == target["correct_choice"]
        assert parsed == (float(row.qa_acc) == 100.0), (path, index)
        if "task" in frame and target["task"] is not None:
            assert str(row.task) == str(target["task"])
    frame["qa_index"] = range(len(frame))
    frame["correct"] = pd.to_numeric(frame.qa_acc).eq(100).astype(int)
    return frame


def fifo_keep_indices(current_lengths, visual_start_idx, kv_size):
    assert visual_start_idx >= 0 and kv_size > 0
    output = []
    for length in current_lengths:
        assert length >= visual_start_idx
        visual = length - visual_start_idx
        keep_visual = min(kv_size, visual)
        prefix = torch.arange(visual_start_idx, dtype=torch.long)
        recent = torch.arange(length - keep_visual, length, dtype=torch.long)
        output.append(torch.cat([prefix, recent]))
    return output


def predict_and_compress_fifo(self):
    """Matched FIFO: prefix + newest visual tokens, no pseudo-query or summary."""
    if self.kv_cache is None:
        return
    from inference.reindex_1d import (compute_cos_sin_for_positions, rotary_delta,
                                      apply_rotary_delta_to_keys_only)
    lengths = [k.shape[2] for k, v in self.kv_cache]
    current_max = max(lengths)
    if current_max > self.kv_size:
        keep_per_layer = fifo_keep_indices(lengths, self.visual_start_idx, self.kv_size)
        device = self.device
        max_pos = getattr(self.language_model.config, "max_position_embeddings", 32768)
        threshold = max_pos - 1024
        current_position = max((p[-1].item() for p in self._position_ids_cache
                                if p is not None and p.numel()), default=0)
        compact = current_position > threshold if self.streaming else True
        old_positions = [p.clone() if p is not None else torch.arange(n, device=device)
                         for p, n in zip(self._position_ids_cache, lengths)]
        new_cache, after = [], []
        for layer, ((keys, values), keep) in enumerate(zip(self.kv_cache, keep_per_layer)):
            keep = keep.to(device=device, dtype=torch.long)
            assert keep.numel() <= lengths[layer]
            kept_k = torch.index_select(keys, 2, keep)
            kept_v = torch.index_select(values, 2, keep)
            old_pos = old_positions[layer].index_select(0, keep)
            if compact:
                new_pos = torch.arange(keep.numel(), device=device, dtype=torch.long)
                cos_old, sin_old = compute_cos_sin_for_positions(
                    self.language_model, len(old_pos), old_pos, keys.dtype, device)
                cos_new, sin_new = compute_cos_sin_for_positions(
                    self.language_model, len(new_pos), new_pos, keys.dtype, device)
                cos_delta, sin_delta = rotary_delta(cos_old, sin_old, cos_new, sin_new)
                kept_k = apply_rotary_delta_to_keys_only(kept_k, cos_delta, sin_delta)
            else:
                new_pos = old_pos
            new_cache.append((kept_k.contiguous(), kept_v.contiguous()))
            self._position_ids_cache[layer] = new_pos.clone().contiguous()
            after.append(kept_k.shape[2])
        self.kv_cache = new_cache
        self.token_activity_cache = [None for _ in range(self.num_layers)]
        expected = self.visual_start_idx + min(self.kv_size, max(lengths) - self.visual_start_idx)
        assert max(after) == min(after) == expected, (after, expected)
        print(f"FIFO_COMPRESSION before_max={current_max} after_min={min(after)} "
              f"after_max={max(after)} budget={self.kv_size} compacted={compact}", flush=True)
    print(f"GPU memory usage: {self.get_gpu_memory_usage_gb()} GB", flush=True)


def _transform_method(model, method_name, replacements):
    """Compile an exact, source-checked method variant and bind only to this instance."""
    import inference.llavaov_hermes as official
    original = inspect.getsource(getattr(official.LlavaOneVision_Hermes, method_name))
    source = textwrap.dedent(original)
    def outside_class_indent(text):
        return "\n".join(line[4:] if line.startswith("    ") else line for line in text.split("\n"))
    for replacement in replacements:
        if len(replacement) == 3:
            start_marker, end_marker, new = replacement
            assert source.count(start_marker) == 1 and source.count(end_marker) == 1
            start = source.index(start_marker)
            end = source.index(end_marker, start)
            source = source[:start] + new + source[end:]
        else:
            old, new = replacement
            old, new = outside_class_indent(old), outside_class_indent(new)
            assert source.count(old) == 1, (method_name, old, source.count(old))
            source = source.replace(old, new)
    namespace = dict(vars(official))
    exec(compile(source, f"<{method_name}_ablation>", "exec"), namespace)
    bound = types.MethodType(namespace[method_name], model)
    setattr(model, method_name, bound)
    return source


def install_ablation(model, method, patch_dir):
    if method == "hermes":
        return []
    if method == "fifo":
        model.predict_and_compress = types.MethodType(predict_and_compress_fifo, model)
        model.fifo_control = True
        return ["predict_and_compress_fifo"]
    written = []
    if method == "no_smoothing":
        source = _transform_method(model, "prune_kv_cache_by_attention", [(
            "    refined_scores = [s.clone() for s in layer_raw_scores]",
            "    for layer_idx, score in enumerate(refined_scores):",
            "    refined_scores = layer_raw_scores  # ABLATION: no cross-layer smoothing\n\n")])
        written.append(("prune_kv_cache_by_attention.py", source))
    elif method == "no_summary":
        source1 = _transform_method(model, "prune_kv_cache_by_attention", [(
            "                layer_budget = max(0, layer_budget - 1)",
            "                layer_budget = layer_budget  # ABLATION: no summary slot reserved")])
        source2 = _transform_method(model, "_shrink_positions_and_rerotate_keys", [(
            "            is_long_term = (layer_idx >= self.long_term_threshold)",
            "            is_long_term = False  # ABLATION: no summary token")])
        written.extend([("prune_kv_cache_by_attention.py", source1),
                        ("_shrink_positions_and_rerotate_keys.py", source2)])
    elif method == "uniform_layers":
        new = """        # ABLATION: common mixed-query/recency policy; retain depth labels only
        # for the independent smoothing and summary-token mechanisms.
        layer_type = ("short-term" if layer_idx < self.short_term_threshold else
                      "long-term" if layer_idx >= self.long_term_threshold else "mid-term")
        layer_attn_weights = attn_weights_mixed[layer_idx]
        question_len = question_len_mixed
        layer_recency_alpha = 0.5
        k = 10.25
"""
        source = _transform_method(model, "prune_kv_cache_by_attention", [
            ("        # Memory hierarchy mode", "        visual_attn_weights =", new)])
        written.append(("prune_kv_cache_by_attention.py", source))
    elif method == "no_position_reindex":
        replacements = [
            ("            if should_compact:\n                new_pos_kept = torch.arange(len(safe_idx), device=device, dtype=torch.long)",
             "            if should_compact and not getattr(self, '_ablate_position_reindex', False):\n                new_pos_kept = torch.arange(len(safe_idx), device=device, dtype=torch.long)"),
            ("            if should_compact:\n                cos_old, sin_old = compute_cos_sin_for_positions(",
             "            if should_compact and not getattr(self, '_ablate_position_reindex', False):\n                cos_old, sin_old = compute_cos_sin_for_positions("),
            ("                    if should_compact:\n                        summary_pos_id = len(safe_idx)",
             "                    if should_compact and not getattr(self, '_ablate_position_reindex', False):\n                        summary_pos_id = len(safe_idx)"),
        ]
        model._ablate_position_reindex = True
        source = _transform_method(model, "_shrink_positions_and_rerotate_keys", replacements)
        written.append(("_shrink_positions_and_rerotate_keys.py", source))
    else:
        raise ValueError(method)
    patch_dir = Path(patch_dir)
    patch_dir.mkdir(parents=True, exist_ok=True)
    for filename, source in written:
        (patch_dir / filename).write_text(source)
    return [name for name, _ in written]


def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def timed_bind(model, attribute, event_name, events):
    original = getattr(model, attribute)
    def timed(*args, **kwargs):
        synchronize()
        start = time.perf_counter()
        try:
            return original(*args, **kwargs)
        finally:
            synchronize()
            event = {"event": event_name, "seconds": time.perf_counter() - start,
                     "allocated_gib": torch.cuda.memory_allocated() / 1024**3,
                     "reserved_gib": torch.cuda.memory_reserved() / 1024**3}
            if event_name == "encode" and args and hasattr(args[0], "shape"):
                event["frames"] = int(args[0].shape[0])
            if event_name in {"prune", "compression_path", "answer"} and getattr(model, "kv_cache", None) is not None:
                lengths = [k.shape[2] for k, v in model.kv_cache]
                event["cache_min"] = min(lengths)
                event["cache_max"] = max(lengths)
            events.append(event)
    setattr(model, attribute, timed)
    return original


def instrument_model(model):
    events, originals = [], {}
    for attribute, name in [("encode_video_chunk", "encode"),
                            ("pseudo_forward", "pseudo_forward"),
                            ("apply_kv_cache_pruning_strict", "prune"),
                            ("question_answering", "answer")]:
        originals[attribute] = timed_bind(model, attribute, name, events)
    return events, originals


def event_totals(events):
    output = {}
    for name in ("encode", "pseudo_forward", "prune", "answer", "compression_path"):
        selected = [float(x["seconds"]) for x in events if x["event"] == name]
        output[name + "_n"] = len(selected)
        output[name + "_s"] = float(sum(selected))
        output[name + "_median_s"] = float(np.median(selected)) if selected else None
        output[name + "_p95_s"] = float(np.quantile(selected, .95)) if selected else None
    output["encoded_frames"] = sum(int(x.get("frames", 0)) for x in events if x["event"] == "encode")
    caches = [int(x["cache_max"]) for x in events if "cache_max" in x]
    output["max_cache_tokens"] = max(caches) if caches else None
    return output


def config_identity(args, annotation, repo):
    source_files = ["inference/llavaov_hermes.py", "inference/abstract_hermes.py",
                    "inference/reindex_1d.py", "video_qa/base.py", "video_qa/hermes_vqa.py"]
    return {
        "method": args.method, "kv_size": args.kv_size, "sample_fps": args.sample_fps,
        "streaming": True, "seed": 2024, "decode": "greedy; repetition_penalty=1.1",
        "model_path_identity": Path(args.model_path).name,
        "annotation_sha256": stable_json(annotation_identity(annotation)).encode(),
        "source_sha256": {name: sha256(Path(repo) / name) for name in source_files},
        "runner_sha256": sha256(__file__),
    }


def run_one_video(analyzer, video, method, output_dir, config_hash):
    model = analyzer.qa_model
    video_id = str(video["video_id"])
    stem = safe_name(video_id)
    video_csv = output_dir / "videos" / f"{stem}.csv"
    metrics_json = output_dir / "video_metrics" / f"{stem}.json"
    if video_csv.is_file() and metrics_json.is_file():
        metrics = json.loads(metrics_json.read_text())
        if metrics.get("config_hash") == config_hash and metrics.get("status") == "complete":
            try:
                frame = validate_video_csv(video_csv, video)
                return frame, metrics, True
            except Exception as exc:
                print(f"REJECT_CHECKPOINT video={video_id} error={type(exc).__name__}; quarantining", flush=True)
    # Reject partial evidence; preserve it outside the active checkpoint namespace.
    incomplete = output_dir / "incomplete" / f"{stem}_{int(time.time())}"
    if video_csv.exists() or metrics_json.exists():
        incomplete.mkdir(parents=True, exist_ok=True)
        if video_csv.exists():
            os.replace(video_csv, incomplete / video_csv.name)
        if metrics_json.exists():
            os.replace(metrics_json, incomplete / metrics_json.name)
    analyzer.record = []
    events, instrumented_originals = instrument_model(model)
    base_predict = model.predict_and_compress
    timed_bind(model, "predict_and_compress", "compression_path", events)
    synchronize()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    started = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    try:
        analyzer.analyze_a_video(video)
        synchronize()
        frame = pd.DataFrame(analyzer.record)
        validate_video_csv(frame.assign(qa_index=range(len(frame))), video) if False else None
        # Validate an actual serialized checkpoint, not an in-memory substitute.
        atomic_csv(video_csv, frame)
        checked = validate_video_csv(video_csv, video)
        elapsed = time.perf_counter() - start
        metrics = {"video_id": video_id, "method": method, "status": "complete",
                   "config_hash": config_hash, "started_utc": started,
                   "wall_s": elapsed, "questions": len(checked),
                   "correct": int(checked.correct.sum()),
                   "accuracy_pct": float(100 * checked.correct.mean()),
                   "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
                   "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
                   "duration_category": video.get("duration_category"),
                   "events": events}
        metrics.update(event_totals(events))
        atomic_json(metrics_json, metrics)
    except Exception:
        # A thrown error leaves no complete marker and therefore cannot be reused.
        raise
    finally:
        model.predict_and_compress = base_predict
        for attribute, original in instrumented_originals.items():
            setattr(model, attribute, original)
        model.clear_cache()
        gc.collect()
        torch.cuda.empty_cache()
    return checked, metrics, False


def run_experiment(args):
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    repo = Path(args.repo).resolve()
    output_dir = Path(args.output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    annotation = json.loads(Path(args.annotation).read_text())
    assert len(annotation) == args.expected_videos
    assert sum(len(x["conversations"]) for x in annotation) == args.expected_questions
    random.seed(2024)
    np.random.seed(2024)
    torch.manual_seed(2024)
    identity = config_identity(args, annotation, repo)
    identity["annotation_sha256"] = hashlib.sha256(identity["annotation_sha256"]).hexdigest()
    config_hash = hashlib.sha256(stable_json(identity).encode()).hexdigest()
    atomic_json(output_dir / "config.json", {**identity, "config_hash": config_hash})
    print("RUN_CONFIG_HASH=" + config_hash, flush=True)
    model, processor = load_model(model_path=args.model_path, kv_size=args.kv_size,
                                  streaming=True, sample_fps=args.sample_fps)
    patches = install_ablation(model, args.method, output_dir / "patches")
    print("INSTANCE_PATCHES=" + json.dumps(patches), flush=True)
    analyzer = HermesVQA(anno=[], save_dir=str(output_dir), sample_fps=args.sample_fps,
                         qa_model=model, qa_processor=processor)
    frames, metrics, reused = [], [], 0
    for index, video in enumerate(annotation):
        frame, metric, was_reused = run_one_video(analyzer, video, args.method, output_dir, config_hash)
        frames.append(frame)
        metrics.append({k: v for k, v in metric.items() if k != "events"})
        reused += int(was_reused)
        print(f"VIDEO_PROGRESS {index + 1}/{len(annotation)} video_id={video['video_id']} "
              f"reused={was_reused} acc={metric['accuracy_pct']:.2f} wall_s={metric['wall_s']:.3f}", flush=True)
    final = pd.concat(frames, ignore_index=True)
    assert len(final) == args.expected_questions
    assert final.video_id.nunique() == args.expected_videos
    atomic_csv(output_dir / "1_0.csv", final.drop(columns=["correct"]))
    atomic_csv(output_dir / "video_summary.csv", pd.DataFrame(metrics))
    complete = {"status": "complete", "method": args.method, "kv_size": args.kv_size,
                "config_hash": config_hash, "videos": len(annotation), "questions": len(final),
                "correct": int(final.correct.sum()), "accuracy_pct": float(100 * final.correct.mean()),
                "reused_videos_this_invocation": reused,
                "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    atomic_json(output_dir / "run_complete.json", complete)
    print("RUN_COMPLETE=" + json.dumps(complete), flush=True)


def selftest():
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        video = {"video_id": "synthetic_only", "conversations": [
            {"question": f"q{i}", "choices": ["a", "b"], "answer": "a"} for i in range(2)]}
        good = pd.DataFrame([{"video_id": "synthetic_only", "question": f"q{i}",
                              "choices": str(["a", "b"]), "answer": "a", "correct_choice": "A",
                              "pred_answer": "A", "pred_choice": "A", "qa_acc": 100} for i in range(2)])
        atomic_csv(root / "good.csv", good)
        assert len(validate_video_csv(root / "good.csv", video)) == 2
        bad = good.copy()
        bad.loc[1, "question"] = "wrong"
        atomic_csv(root / "bad.csv", bad)
        try:
            validate_video_csv(root / "bad.csv", video)
            raise AssertionError("malformed evidence was accepted")
        except AssertionError:
            pass
        indices = fifo_keep_indices([14, 30, 114], 14, 10)
        assert [x.tolist() for x in indices] == [list(range(14)), list(range(14)) + list(range(20, 30)),
                                                list(range(14)) + list(range(104, 114))]
        atomic_json(root / "atomic.json", {"ok": True})
        assert json.loads((root / "atomic.json").read_text())["ok"]
    print("SELFTEST_PASS: synthetic fixtures only; no research result written", flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--selftest", action="store_true")
    parser.add_argument("--repo")
    parser.add_argument("--annotation")
    parser.add_argument("--output_dir")
    parser.add_argument("--model_path")
    parser.add_argument("--method", choices=["hermes", "fifo", "no_smoothing", "no_summary",
                                               "uniform_layers", "no_position_reindex"])
    parser.add_argument("--kv_size", type=int)
    parser.add_argument("--sample_fps", type=float, default=.5)
    parser.add_argument("--expected_videos", type=int)
    parser.add_argument("--expected_questions", type=int)
    args = parser.parse_args()
    if args.selftest:
        selftest()
        return
    assert args.expected_videos and args.expected_questions and args.kv_size
    run_experiment(args)


if __name__ == "__main__":
    main()
''')
ATTENTION_SCRIPT=SCRIPTS/'validate_real_video_attention.py'
ATTENTION_SCRIPT.write_text(r'''#!/usr/bin/env python3
"""Differential pseudo-query attention on a real 16-frame visual KV cache."""
import argparse
import gc
import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers.cache_utils import DynamicCache


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def load_first_chunk(video_path, sample_fps=.5, frames=16):
    reader = VideoReader(video_path, num_threads=1)
    fps = round(reader.get_avg_fps())
    step = int(fps / sample_fps)
    assert step > 0
    indices = list(range(0, len(reader), step))[:frames]
    assert len(indices) == frames, (video_path, len(indices))
    return torch.from_numpy(reader.get_batch(indices).asnumpy())


def manual_attention(model, input_ids, past_key_values, causal):
    """Current frozen-hidden-state computation, optionally with causal masking."""
    from inference.reindex_1d import _get_rotary_module, apply_rotary_pos_emb_1d
    starts = model._get_next_start_pos_per_layer()
    q_len, batch = input_ids.shape[1], input_ids.shape[0]
    hidden_states = model.get_input_embeddings()(input_ids)
    config = model.language_model.config
    heads, kv_heads = config.num_attention_heads, config.num_key_value_heads
    head_dim = config.hidden_size // heads
    output = []
    for layer_index in range(config.num_hidden_layers):
        layer = model.language_model.model.layers[layer_index]
        past_k, past_v = past_key_values[layer_index]
        positions = model._build_position_ids(starts[layer_index], q_len, batch)
        normalized = layer.input_layernorm(hidden_states)
        attention = layer.self_attn
        query = attention.q_proj(normalized).view(batch, q_len, heads, head_dim).transpose(1, 2)
        key = attention.k_proj(normalized).view(batch, q_len, kv_heads, head_dim).transpose(1, 2)
        value = attention.v_proj(normalized).view(batch, q_len, kv_heads, head_dim).transpose(1, 2)
        rotary = _get_rotary_module(model.language_model)
        cos, sin = rotary(value, positions.max().item() + 1)
        cos, sin = cos[positions].unsqueeze(1), sin[positions].unsqueeze(1)
        query = apply_rotary_pos_emb_1d(query, cos, sin)
        key = apply_rotary_pos_emb_1d(key, cos, sin)
        key = torch.cat([past_k, key], dim=2)
        if kv_heads != heads:
            key = torch.repeat_interleave(key, heads // kv_heads, dim=1)
        weights = torch.matmul(query.float(), key.float().transpose(-2, -1)) / math.sqrt(head_dim)
        if causal:
            past_len = past_k.shape[2]
            causal_mask = torch.ones(q_len, q_len, dtype=torch.bool, device=weights.device).tril()
            allowed = torch.cat([torch.ones(q_len, past_len, dtype=torch.bool, device=weights.device),
                                 causal_mask], dim=1)
            weights = weights.masked_fill(~allowed, torch.finfo(weights.dtype).min)
        output.append(F.softmax(weights, dim=-1, dtype=torch.float32).to(query.dtype))
    return output


def native_attention(model, input_ids):
    starts = model._get_next_start_pos_per_layer()
    q_len, batch = input_ids.shape[1], input_ids.shape[0]
    model._layer_position_ids.clear()
    for layer in range(model.num_layers):
        model._layer_position_ids[layer] = model._build_position_ids(starts[layer], q_len, batch)
    positions = model._build_position_ids(starts[0], q_len, batch)
    model._set_rotary_required_len(max(starts) + q_len)
    legacy = tuple((k.clone(), v.clone()) for k, v in model.kv_cache)
    output = model.language_model(
        input_ids=input_ids, use_cache=False,
        past_key_values=DynamicCache.from_legacy_cache(legacy),
        output_attentions=True, position_ids=positions,
        cache_position=None, attention_mask=None)
    model._layer_position_ids.clear()
    assert output.attentions is not None and len(output.attentions) == model.num_layers
    return list(output.attentions)


def current_attention(model, input_ids):
    return model._compute_attention_scores_manually(input_ids, model.kv_cache)


def compare(path_a, path_b, query, a_name, b_name):
    rows = []
    assert len(path_a) == len(path_b)
    for layer, (a, b) in enumerate(zip(path_a, path_b)):
        assert a.shape == b.shape and a.shape[0] == 1
        x, y = a[0].float().flatten(), b[0].float().flatten()
        cosine = F.cosine_similarity(x, y, dim=0).item()
        difference = (x - y).abs()
        rows.append({"query": query, "comparison": f"{a_name}_vs_{b_name}", "layer": layer,
                     "cosine": cosine, "mean_absolute_difference": difference.mean().item(),
                     "max_absolute_difference": difference.max().item(),
                     "shape": list(a.shape)})
    return rows


def selection_indices(model, attentions, budget):
    local, global_, mixed = [attentions[name] for name in ("local", "global", "mixed")]
    model.token_activity_cache = [None for _ in range(model.num_layers)]
    keep = model.prune_kv_cache_by_attention(local, global_, mixed, num_keep=budget)
    return keep


def compare_selection(model, paths, budget):
    selected = {name: selection_indices(model, value, budget) for name, value in paths.items()}
    visual_count = model.kv_cache[0][0].shape[2] - model.visual_start_idx
    assert visual_count > budget, (visual_count, budget)
    rows = []
    for comparison, a_name, b_name in [("current_vs_native", "current", "native"),
                                       ("causal_frozen_vs_native", "causal_frozen", "native"),
                                       ("current_vs_causal_frozen", "current", "causal_frozen")]:
        for layer, (a, b) in enumerate(zip(selected[a_name], selected[b_name])):
            a = set(a) - set(range(model.visual_start_idx))
            b = set(b) - set(range(model.visual_start_idx))
            assert 0 < len(a) < visual_count and 0 < len(b) < visual_count
            rows.append({"comparison": comparison, "layer": layer, "budget": budget,
                         "visual_tokens": visual_count, "selected_A": len(a), "selected_B": len(b),
                         "intersection": len(a & b), "jaccard": len(a & b) / len(a | b),
                         "symmetric_difference": len(a ^ b),
                         "selected_indices_A": json.dumps(sorted(a)),
                         "selected_indices_B": json.dumps(sorted(b))})
    return rows


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo", required=True)
    parser.add_argument("--model_path", required=True)
    parser.add_argument("--video_path", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--budget", type=int, default=256)
    args = parser.parse_args()
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    from inference.llavaov_hermes import load_model
    torch.manual_seed(2024)
    np.random.seed(2024)
    model, processor = load_model(model_path=args.model_path, kv_size=args.budget,
                                  streaming=True, sample_fps=.5)
    video = load_first_chunk(args.video_path)
    model.encode_init_prompt()
    model.encode_video_chunk(video)
    assert isinstance(model.kv_cache, tuple)
    past_len = model.kv_cache[0][0].shape[2]
    visual_tokens = past_len - model.visual_start_idx
    assert visual_tokens > args.budget and visual_tokens > 1000
    local, global_ = model.predict_next_question()
    queries = {"local": local, "global": global_, "mixed": local + "; " + global_}
    paths = {}
    for path_name in ("native", "current", "causal_frozen"):
        paths[path_name] = {}
        for query_name, text in queries.items():
            ids = torch.as_tensor([processor.tokenizer(text).input_ids], device=model.device, dtype=torch.int)
            print(f"ATTENTION_START path={path_name} query={query_name} q_len={ids.shape[1]}", flush=True)
            if path_name == "native":
                weights = native_attention(model, ids)
            elif path_name == "current":
                weights = current_attention(model, ids)
            else:
                weights = manual_attention(model, ids, model.kv_cache, causal=True)
            paths[path_name][query_name] = weights
            print(f"ATTENTION_DONE path={path_name} query={query_name}", flush=True)
    metric_rows = []
    for query in queries:
        for comparison, a, b in [("current_vs_native", "current", "native"),
                                 ("causal_frozen_vs_native", "causal_frozen", "native"),
                                 ("current_vs_causal_frozen", "current", "causal_frozen")]:
            metric_rows.extend(compare(paths[a][query], paths[b][query], query, a, b))
    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(output / "attention_per_layer_metrics.csv", index=False)
    summary = metrics.groupby("comparison").agg(
        layers=("layer", "size"), mean_cosine=("cosine", "mean"), min_cosine=("cosine", "min"),
        mean_absolute_difference=("mean_absolute_difference", "mean"),
        max_absolute_difference=("max_absolute_difference", "max")).reset_index()
    summary.to_csv(output / "attention_summary.csv", index=False)
    selection = pd.DataFrame(compare_selection(model, paths, args.budget))
    selection.to_csv(output / "selection_impact_per_layer.csv", index=False)
    selection.groupby("comparison").agg(layers=("layer", "size"), mean_jaccard=("jaccard", "mean"),
        min_jaccard=("jaccard", "min"), mean_symmetric_difference=("symmetric_difference", "mean")).reset_index().to_csv(
        output / "selection_impact_summary.csv", index=False)
    provenance = {"status": "complete", "video_path_name": Path(args.video_path).name,
                  "video_sha256": sha256(args.video_path), "encoded_frames": 16,
                  "past_tokens": past_len, "visual_tokens": visual_tokens,
                  "visual_start_idx": model.visual_start_idx, "layers": model.num_layers,
                  "topk_budget_per_layer": args.budget, "topk_is_constraining": True,
                  "queries": queries, "query_tokens": {name: len(processor.tokenizer(text).input_ids)
                                                       for name, text in queries.items()},
                  "current": "frozen embeddings; no causal mask",
                  "causal_frozen": "frozen embeddings; causal mask",
                  "native": "causal residual/MLP propagation; DynamicCache reference",
                  "model_path_identity": Path(args.model_path).name,
                  "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    (output / "attention_test_provenance.json").write_text(json.dumps(provenance, indent=2))
    (output / "complete.json").write_text(json.dumps({"status": "complete", "metric_rows": len(metrics),
                                                       "selection_rows": len(selection)}))
    print(summary.to_string(index=False), flush=True)
    print("REAL_VIDEO_ATTENTION_COMPLETE", flush=True)
    del paths, model
    gc.collect()
    torch.cuda.empty_cache()


if __name__ == "__main__":
    main()
''')
EFFICIENCY_SCRIPT=SCRIPTS/'efficiency_interleaved.py'
EFFICIENCY_SCRIPT.write_text(r'''#!/usr/bin/env python3
"""Same-session, video-interleaved HERMES/FIFO efficiency comparison."""
import argparse
import importlib.util
import json
import random
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def load_runner(path):
    spec = importlib.util.spec_from_file_location("experiment_runner", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo", required=True)
    parser.add_argument("--runner", required=True)
    parser.add_argument("--annotation", required=True)
    parser.add_argument("--model_path", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--videos", type=int, default=10)
    parser.add_argument("--repetitions", type=int, default=3)
    parser.add_argument("--kv_size", type=int, default=4000)
    args = parser.parse_args()
    assert args.videos == 10 and args.repetitions == 3 and args.kv_size == 4000
    R = load_runner(args.runner)
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    annotation = json.loads(Path(args.annotation).read_text())[:args.videos]
    assert len(annotation) == 10 and sum(len(x["conversations"]) for x in annotation) == 50
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    random.seed(2024)
    np.random.seed(2024)
    torch.manual_seed(2024)
    model, processor = load_model(model_path=args.model_path, kv_size=args.kv_size,
                                  streaming=True, sample_fps=.5)
    hermes_predict = model.predict_and_compress
    fifo_predict = types.MethodType(R.predict_and_compress_fifo, model)
    analyzer = HermesVQA(anno=[], save_dir=str(output), sample_fps=.5,
                         qa_model=model, qa_processor=processor)
    order_path = output / "execution_order.csv"
    order = pd.read_csv(order_path).to_dict("records") if order_path.is_file() else []
    order_keys = {(int(x["repetition"]), str(x["video_id"]), x["method"]) for x in order}
    for repetition in range(args.repetitions):
        for video_index, video in enumerate(annotation):
            # ABBA-like ordering: reverse treatment order on alternating videos/repetitions.
            methods = ["hermes", "fifo"] if (repetition + video_index) % 2 == 0 else ["fifo", "hermes"]
            for method in methods:
                key = (repetition, str(video["video_id"]), method)
                target = output / f"repeat{repetition}" / method / R.safe_name(video["video_id"])
                target.mkdir(parents=True, exist_ok=True)
                csv_path, json_path = target / "predictions.csv", target / "metrics.json"
                if csv_path.is_file() and json_path.is_file():
                    metric = json.loads(json_path.read_text())
                    if metric.get("status") == "complete":
                        R.validate_video_csv(csv_path, video)
                        print(f"REUSE_INTERLEAVED rep={repetition} video={video['video_id']} method={method}", flush=True)
                        continue
                for path in (csv_path, json_path):
                    if path.exists():
                        path.rename(path.with_suffix(path.suffix + f".incomplete_{int(time.time())}"))
                model.predict_and_compress = hermes_predict if method == "hermes" else fifo_predict
                analyzer.record = []
                events, originals = R.instrument_model(model)
                base_predict = model.predict_and_compress
                R.timed_bind(model, "predict_and_compress", "compression_path", events)
                R.synchronize()
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                start = time.perf_counter()
                started = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
                try:
                    analyzer.analyze_a_video(video)
                    R.synchronize()
                    frame = pd.DataFrame(analyzer.record)
                    R.atomic_csv(csv_path, frame)
                    checked = R.validate_video_csv(csv_path, video)
                    elapsed = time.perf_counter() - start
                    metric = {"status": "complete", "repetition": repetition,
                              "video_index": video_index, "video_id": str(video["video_id"]),
                              "method": method, "kv_size": args.kv_size, "started_utc": started,
                              "wall_s": elapsed, "questions": len(checked),
                              "correct": int(checked.correct.sum()),
                              "accuracy_pct": float(100 * checked.correct.mean()),
                              "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
                              "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
                              "events": events}
                    metric.update(R.event_totals(events))
                    R.atomic_json(json_path, metric)
                    order.append({"repetition": repetition, "video_index": video_index,
                                  "video_id": str(video["video_id"]), "method": method,
                                  "started_utc": started, "wall_s": elapsed})
                    R.atomic_csv(order_path, pd.DataFrame(order))
                    print(f"INTERLEAVED_COMPLETE rep={repetition} video={video['video_id']} "
                          f"method={method} wall_s={elapsed:.3f}", flush=True)
                finally:
                    model.predict_and_compress = base_predict
                    for attribute, original in originals.items():
                        setattr(model, attribute, original)
                    model.clear_cache()
    complete = []
    for repetition in range(args.repetitions):
        for video in annotation:
            for method in ("hermes", "fifo"):
                target = output / f"repeat{repetition}" / method / R.safe_name(video["video_id"])
                assert json.loads((target / "metrics.json").read_text())["status"] == "complete"
                R.validate_video_csv(target / "predictions.csv", video)
                complete.append((repetition, str(video["video_id"]), method))
    assert len(complete) == 60 and len(set(complete)) == 60
    R.atomic_json(output / "complete.json", {"status": "complete", "video_runs": 60,
                  "videos": 10, "repetitions": 3, "methods": ["hermes", "fifo"], "kv_size": 4000})
    print("INTERLEAVED_EFFICIENCY_COMPLETE=60_video_runs", flush=True)


if __name__ == "__main__":
    main()
''')

for script in (RUNNER,ATTENTION_SCRIPT,EFFICIENCY_SCRIPT):
    compile(script.read_text(),str(script),'exec');print('SCRIPT SYNTAX PASS',script,sha256(script))
print(run([PY312,str(RUNNER),'--selftest'],env=SUBENV).stdout)
# Apply every ablation transform to a dummy instance before any long run.
ablation_probe=r"""
import importlib.util,sys,types,tempfile
from pathlib import Path
spec=importlib.util.spec_from_file_location('runner',sys.argv[1]);r=importlib.util.module_from_spec(spec);spec.loader.exec_module(r)
for method in ('fifo','no_smoothing','no_summary','uniform_layers','no_position_reindex'):
    dummy=types.SimpleNamespace()
    with tempfile.TemporaryDirectory() as out:
        patches=r.install_ablation(dummy,method,out)
        for path in Path(out).glob('*.py'):compile(path.read_text(),str(path),'exec')
    print('ABLATION_SOURCE_TRANSFORM_PASS',method,patches)
"""
print(run([PY312,'-c',ablation_probe,str(RUNNER)],cwd=ROOT,env=SUBENV).stdout)
print('All script, checkpoint, FIFO, and ablation static/synthetic tests passed.')

## 4. Stage 1 — real-video attention validation

One deterministic 16-frame chunk from sample 201 creates a real visual cache. Current manual, causal-frozen, and native propagated attention are compared per layer. Retention uses a constraining 256-token visual budget and reports actual selected-index Jaccard/symmetric differences. Complete outputs are reused; partial outputs stop for inspection.

In [ ]:
records201,videos201,annotation201=prepare_shard(201,250)
ATTN_OUT=EXPORTS/'real_video_attention';ATTN_OUT.mkdir(parents=True,exist_ok=True)
attention_required=['attention_per_layer_metrics.csv','attention_summary.csv','selection_impact_per_layer.csv',
                    'selection_impact_summary.csv','attention_test_provenance.json','complete.json']
attention_complete=all((ATTN_OUT/x).is_file() for x in attention_required)
if attention_complete:
    prov=json.loads((ATTN_OUT/'attention_test_provenance.json').read_text())
    metrics=pd.read_csv(ATTN_OUT/'attention_per_layer_metrics.csv');selection=pd.read_csv(ATTN_OUT/'selection_impact_per_layer.csv')
    assert prov['status']=='complete' and prov['visual_tokens']>prov['topk_budget_per_layer']
    assert len(metrics)==3*3*24 and len(selection)==3*24
    print('Reusing complete real-video attention outputs.')
else:
    assert not any(ATTN_OUT.glob('*')),'Partial attention outputs found; move them aside for inspection before rerunning.'
    run_logged([PY312,str(ATTENTION_SCRIPT),'--repo',str(ROOT),'--model_path',str(MODEL_SNAPSHOT),
                '--video_path',records201[0]['video_path'],'--output_dir',str(ATTN_OUT),'--budget','256'],
               LOGS/'real_video_attention.log',env=SUBENV)
    attention_complete=all((ATTN_OUT/x).is_file() for x in attention_required);assert attention_complete
attention_summary=pd.read_csv(ATTN_OUT/'attention_summary.csv')
selection_summary=pd.read_csv(ATTN_OUT/'selection_impact_summary.csv')
display(attention_summary.round(5));display(selection_summary.round(4))

## 5. Stage 2 — non-overlapping generalization on samples 251–300

All four matched 4K/6K HERMES/FIFO runs are mandatory. The 50K near-uncompressed HERMES control is attempted without truncation; a subprocess failure is recorded explicitly and does not become a zero-row accuracy result.

In [ ]:
records251,videos251,annotation251=prepare_shard(251,300)
GENERAL=RESULTS/'generalization_251_300'
GENERAL_METHODS=[('hermes',4000),('hermes',6000),('fifo',4000),('fifo',6000),('hermes',50000)]
def run_dir(base,method,kv):return base/f'{method}_kv{kv}'
def complete_run(path,records):
    complete=path/'run_complete.json';config=path/'config.json';csv=path/'1_0.csv'
    if not (complete.is_file() and config.is_file() and csv.is_file()):return None
    status=json.loads(complete.read_text());identity=json.loads(config.read_text())
    assert status['status']=='complete' and status['config_hash']==identity['config_hash']
    assert status['method']==path.name.split('_kv')[0] and int(status['kv_size'])==int(path.name.split('_kv')[1])
    assert status['videos']==len(records) and status['questions']==sum(len(x['conversations']) for x in records)
    for record in records:
        stem=re.sub(r'[^A-Za-z0-9_.-]+','_',str(record['video_id']))
        metric=json.loads((path/'video_metrics'/f'{stem}.json').read_text())
        assert metric['status']=='complete' and metric['config_hash']==identity['config_hash']
        read_predictions(path/'videos'/f'{stem}.csv',[record],verbose=False)
    return read_predictions(csv,records,path.parent.name+'/'+path.name)
def launch_experiment(base,method,kv,records,annotation_path,optional=False):
    path=run_dir(base,method,kv);existing=complete_run(path,records)
    if existing is not None:
        print('REUSE COMPLETE RUN',method,kv);return existing,'reused'
    try:
        run_logged([PY312,str(RUNNER),'--repo',str(ROOT),'--annotation',str(annotation_path),
            '--output_dir',str(path),'--model_path',str(MODEL_SNAPSHOT),'--method',method,'--kv_size',str(kv),
            '--sample_fps',str(FPS),'--expected_videos',str(len(records)),'--expected_questions',
            str(sum(len(x['conversations']) for x in records))],LOGS/f'{path.parent.name}_{path.name}.log',env=SUBENV)
        result=complete_run(path,records);assert result is not None
        return result,'completed'
    except Exception as exc:
        if not optional:raise
        failure={'method':method,'kv_size':kv,'status':'failed_feasibility_control','error_type':type(exc).__name__,
                 'log':str((LOGS/f'{path.parent.name}_{path.name}.log').name),
                 'finished_utc':datetime.now(timezone.utc).isoformat(),'input_truncated':False}
        path.mkdir(parents=True,exist_ok=True)
        (path/'run_failed.json').write_text(json.dumps(failure,indent=2))
        print('EXPLICIT OPTIONAL CONTROL FAILURE:',method,kv,type(exc).__name__)
        return None,'failed'
GENERAL_RESULTS={};GENERAL_STATUS=[]
for method,kv in GENERAL_METHODS:
    result,status=launch_experiment(GENERAL,method,kv,records251,annotation251,optional=(kv==50000))
    name=f'{method.upper()}_{kv}';GENERAL_STATUS.append({'name':name,'status':status})
    if result is not None:GENERAL_RESULTS[name]=result
pd.DataFrame(GENERAL_STATUS).to_csv(EXPORTS/'generalization_run_status.csv',index=False)
assert all(name in GENERAL_RESULTS for name in ('HERMES_4000','HERMES_6000','FIFO_4000','FIFO_6000'))
print('Generalization mandatory runs complete and validated.')

## 6. Stage 3 — one-factor component ablations at 4K

The variants are: no cross-layer smoothing; no summary tokens (including no reserved summary slot); one uniform mixed-query/recency policy across depths; and no position re-indexing when compaction would otherwise occur. Only loaded model instances are patched. Exact source-transform tests ran before inference.

In [ ]:
ABLATIONS=RESULTS/'ablations_251_300';ABLATION_RESULTS={}
for method in ('no_smoothing','no_summary','uniform_layers','no_position_reindex'):
    result,status=launch_experiment(ABLATIONS,method,4000,records251,annotation251)
    ABLATION_RESULTS[method.upper()+'_4000']=result
print('All four component ablations complete and validated.')

## 7. Stage 4 — same-session interleaved efficiency

Ten videos from the new shard are evaluated for three repetitions. Treatment order alternates by video and repetition. Each video run records total wall time, answer/encoding/pseudo-query/pruning events, peak allocated and reserved memory, and raw predictions. These are repeated measurements on the same videos, not independent accuracy samples.

In [ ]:
EFF_OUT=WORK/'efficiency_interleaved'
if (EFF_OUT/'complete.json').is_file():
    status=json.loads((EFF_OUT/'complete.json').read_text());assert status['video_runs']==60
    print('Reusing 60 complete interleaved video runs.')
else:
    run_logged([PY312,str(EFFICIENCY_SCRIPT),'--repo',str(ROOT),'--runner',str(RUNNER),
        '--annotation',str(annotation251),'--model_path',str(MODEL_SNAPSHOT),'--output_dir',str(EFF_OUT),
        '--videos','10','--repetitions','3','--kv_size','4000'],LOGS/'efficiency_interleaved.log',env=SUBENV)
    assert json.loads((EFF_OUT/'complete.json').read_text())['video_runs']==60
# Re-read every raw prediction and timing record.
eff_rows=[];eff_event_rows=[]
for rep in range(3):
    for video in records251[:10]:
        for method in ('hermes','fifo'):
            target=EFF_OUT/f'repeat{rep}'/method/re.sub(r'[^A-Za-z0-9_.-]+','_',str(video['video_id']))
            pred=read_predictions(target/'predictions.csv',[video],f'efficiency rep{rep} {method} {video["video_id"]}')
            metric=json.loads((target/'metrics.json').read_text());assert metric['status']=='complete'
            eff_rows.append({k:v for k,v in metric.items() if k!='events'})
            for event in metric['events']:
                eff_event_rows.append({'repetition':rep,'video_id':str(video['video_id']),'method':method,**event})
assert len(eff_rows)==60
pd.DataFrame(eff_rows).to_csv(EXPORTS/'efficiency_interleaved_video_metrics.csv',index=False)
pd.DataFrame(eff_event_rows).to_csv(EXPORTS/'efficiency_interleaved_event_metrics.csv',index=False)
print('Interleaved efficiency evidence validated: 60/60 video runs.')

## 8. Stage 5 — resumable full real-time benchmark

The primary full-benchmark contrast is HERMES-4K versus FIFO-4K. Samples 201–250 come from immutable prior evidence; 251–300 reuse Stage 2. Every other 50-video shard is downloaded, hash-checked, evaluated, and checkpointed. A shard directory can be cleaned only after both methods are complete. The final aggregate is accepted only at exactly 498 videos and 2,495 questions.

In [ ]:
FULL=RESULTS/'full_benchmark';FULL.mkdir(parents=True,exist_ok=True)
FULL_STATUS=[]
# Preserve an explicit record for the already-completed shard; never launch it.
for method in ('hermes','fifo'):
    name='HERMES_4000' if method=='hermes' else 'FIFO_4000'
    FULL_STATUS.append({'shard':'201-250','method':method,'kv':4000,'status':'reused_prior','rows':len(PRIOR[name])})
for start,end, *_ in SHARDS:
    if start==201:continue
    annotation_path=DATA/f'streamingbench_{start}_{end}.json'
    records=json.loads(annotation_path.read_text()) if annotation_path.is_file() else None
    video_dir=None
    if start==251:
        # Stage 2 outputs are the canonical full-benchmark outputs for this shard.
        for method,name in (('hermes','HERMES_4000'),('fifo','FIFO_4000')):
            result=GENERAL_RESULTS[name]
            FULL_STATUS.append({'shard':f'{start}-{end}','method':method,'kv':4000,'status':'reused_generalization','rows':len(result)})
        continue
    # Complete shard outputs are recognized without re-downloading videos that were cleaned.
    needs_videos=records is None or any(complete_run(run_dir(FULL/f'{start}_{end}',method,4000),records) is None
                                       for method in ('hermes','fifo'))
    if needs_videos:records,video_dir,annotation_path=prepare_shard(start,end)
    for method in ('hermes','fifo'):
        result,status=launch_experiment(FULL/f'{start}_{end}',method,4000,records,annotation_path)
        FULL_STATUS.append({'shard':f'{start}-{end}','method':method,'kv':4000,'status':status,'rows':len(result)})
    if video_dir is not None:
        shutil.rmtree(video_dir)
        print('Cleaned completed video directory:',video_dir)
pd.DataFrame(FULL_STATUS).to_csv(EXPORTS/'full_benchmark_shard_status.csv',index=False)
FULL_RESULTS={}
for method,prior_name in (('hermes','HERMES_4000'),('fifo','FIFO_4000')):
    pieces=[PRIOR[prior_name].copy()]
    for start,end, *_ in SHARDS:
        if start==201:continue
        if start==251:
            pieces.append(GENERAL_RESULTS['HERMES_4000' if method=='hermes' else 'FIFO_4000'].copy())
            continue
        path=run_dir(FULL/f'{start}_{end}',method,4000)
        records=json.loads((DATA/f'streamingbench_{start}_{end}.json').read_text())
        pieces.append(read_predictions(path/'1_0.csv',records,f'full {method} {start}-{end}'))
    combined=pd.concat(pieces,ignore_index=True)
    assert len(combined)==2495 and combined.video_id.nunique()==498
    assert not combined.duplicated(KEY).any()
    expected_keys={(str(x['video_id']),i) for x in ANNOTATIONS for i in range(len(x['conversations']))}
    assert set(map(tuple,combined[KEY].values))==expected_keys
    combined.to_csv(EXPORTS/f'full_{method}_4000_predictions.csv',index=False)
    FULL_RESULTS[method.upper()+'_4000']=combined
print('FULL BENCHMARK COMPLETE: both methods, 498 videos, 2495 questions each.')

In [ ]:
def paired(a,b,name_a,name_b,reps=BOOTSTRAP_REPS,seed=SEED):
    aa=a.set_index(KEY).sort_index();bb=b.set_index(KEY).sort_index();assert aa.index.equals(bb.index)
    for field in ('question','choices','correct_choice','answer'):
        assert aa[field].fillna('').astype(str).str.strip().equals(bb[field].fillna('').astype(str).str.strip()),field
    x=a[KEY+['correct']].rename(columns={'correct':'a'}).merge(b[KEY+['correct']].rename(columns={'correct':'b'}),
         on=KEY,validate='one_to_one')
    n10=int(((x.a==1)&(x.b==0)).sum());n01=int(((x.a==0)&(x.b==1)).sum())
    by_video=x.assign(delta=x.a-x.b).groupby('video_id').delta.agg(['sum','count'])
    sums,counts=by_video['sum'].to_numpy(float),by_video['count'].to_numpy(float)
    rng=np.random.default_rng(seed);draws=rng.integers(0,len(sums),size=(reps,len(sums)))
    boot=100*sums[draws].sum(axis=1)/counts[draws].sum(axis=1)
    return {'A':name_a,'B':name_b,'n':len(x),'n_videos':x.video_id.nunique(),
        'accuracy_A_pct':100*x.a.mean(),'accuracy_B_pct':100*x.b.mean(),
        'difference_A_minus_B_pp':100*(x.a-x.b).mean(),'A_only_correct':n10,'B_only_correct':n01,
        'video_cluster_CI95_low_pp':float(np.quantile(boot,.025)),
        'video_cluster_CI95_high_pp':float(np.quantile(boot,.975)),
        'exact_discordance_p_exploratory':binomtest(n10,n10+n01,.5).pvalue if n10+n01 else 1.0}
def accuracy_rows(results,scope):
    return [{'scope':scope,'method':name,'videos':d.video_id.nunique(),'questions':len(d),
             'correct':int(d.correct.sum()),'accuracy_pct':100*d.correct.mean()} for name,d in results.items()]
def markdown_table(frame):
    d=frame.copy();d.columns=[str(x) for x in d.columns]
    lines=['| '+' | '.join(d.columns)+' |','| '+' | '.join(['---']*len(d.columns))+' |']
    for row in d.itertuples(index=False,name=None):lines.append('| '+' | '.join(str(x) for x in row)+' |')
    return '\n'.join(lines)

In [ ]:
# Accuracy tables contain only actually completed runs.
accuracy=pd.DataFrame(accuracy_rows(PRIOR,'prior_201_250')+
                      accuracy_rows(GENERAL_RESULTS,'generalization_251_300')+
                      accuracy_rows(ABLATION_RESULTS,'ablation_251_300')+
                      accuracy_rows(FULL_RESULTS,'full_498'))
accuracy.to_csv(EXPORTS/'master_accuracy_all_stages.csv',index=False)
comparisons=[]
for kv in (4000,6000):
    comparisons.append(paired(GENERAL_RESULTS[f'HERMES_{kv}'],GENERAL_RESULTS[f'FIFO_{kv}'],
                              f'HERMES_{kv}',f'FIFO_{kv}'))
if 'HERMES_50000' in GENERAL_RESULTS:
    for kv in (4000,6000):comparisons.append(paired(GENERAL_RESULTS[f'HERMES_{kv}'],GENERAL_RESULTS['HERMES_50000'],
                                                   f'HERMES_{kv}','HERMES_50000'))
paired_general=pd.DataFrame(comparisons);paired_general.to_csv(EXPORTS/'generalization_paired.csv',index=False)
ablation_pairs=pd.DataFrame([paired(GENERAL_RESULTS['HERMES_4000'],d,'HERMES_4000',name)
                             for name,d in ABLATION_RESULTS.items()])
ablation_pairs.to_csv(EXPORTS/'ablation_paired.csv',index=False)
full_pair=pd.DataFrame([paired(FULL_RESULTS['HERMES_4000'],FULL_RESULTS['FIFO_4000'],'HERMES_4000','FIFO_4000')])
full_pair.to_csv(EXPORTS/'full_benchmark_paired.csv',index=False)
# Per-task denominators remain visible; no macro-average claim is substituted for micro accuracy.
task_tables=[]
for scope,results in (('generalization',GENERAL_RESULTS),('ablations',ABLATION_RESULTS),('full',FULL_RESULTS)):
    for name,d in results.items():
        if 'task' in d:
            table=d.groupby('task').correct.agg(['size','sum','mean']).reset_index()
            table.columns=['task','questions','correct','accuracy'];table['accuracy_pct']=100*table.pop('accuracy')
            table.insert(0,'method',name);table.insert(0,'scope',scope);task_tables.append(table)
pd.concat(task_tables,ignore_index=True).to_csv(EXPORTS/'accuracy_by_task.csv',index=False)
# Interleaved summaries and paired per-video timing differences.
eff=pd.DataFrame(eff_rows)
eff_summary=eff.groupby('method').agg(video_runs=('video_id','size'),questions=('questions','sum'),
    correct=('correct','sum'),median_video_wall_s=('wall_s','median'),p95_video_wall_s=('wall_s',lambda x:x.quantile(.95)),
    total_wall_s=('wall_s','sum'),median_answer_s=('answer_median_s','median'),
    total_encode_s=('encode_s','sum'),encoded_frames=('encoded_frames','sum'),
    total_compression_path_s=('compression_path_s','sum'),total_pseudo_forward_s=('pseudo_forward_s','sum'),
    total_prune_s=('prune_s','sum'),max_cache_tokens=('max_cache_tokens','max'),median_peak_allocated_gib=('peak_allocated_gib','median'),
    max_peak_allocated_gib=('peak_allocated_gib','max'),median_peak_reserved_gib=('peak_reserved_gib','median'),
    max_peak_reserved_gib=('peak_reserved_gib','max')).reset_index()
eff_summary['accuracy_pct']=100*eff_summary.correct/eff_summary.questions
eff_summary['frames_per_encode_s']=eff_summary.encoded_frames/eff_summary.total_encode_s
answer_events=pd.DataFrame(eff_event_rows);answer_events=answer_events[answer_events.event.eq('answer')]
answer_summary=answer_events.groupby('method').seconds.agg(['size','median','mean',
    lambda x:x.quantile(.95),lambda x:x.quantile(.99)]).reset_index()
answer_summary.columns=['method','answer_events','median_answer_s','mean_answer_s','p95_answer_s','p99_answer_s']
answer_summary.to_csv(EXPORTS/'efficiency_answer_latency_summary.csv',index=False)
eff_summary.to_csv(EXPORTS/'efficiency_interleaved_summary.csv',index=False)
wide=eff.pivot(index=['repetition','video_id'],columns='method',values='wall_s').reset_index()
wide['hermes_minus_fifo_s']=wide.hermes-wide.fifo
wide['fifo_over_hermes_ratio']=wide.fifo/wide.hermes
wide.to_csv(EXPORTS/'efficiency_interleaved_paired_video.csv',index=False)
# Figures are generated from validated outputs only.
plt.figure(figsize=(10,5))
plot_data=accuracy[accuracy.scope.isin(['generalization_251_300','ablation_251_300','full_498'])]
plt.bar(range(len(plot_data)),plot_data.accuracy_pct);plt.xticks(range(len(plot_data)),plot_data.method,rotation=70,ha='right')
plt.ylabel('Accuracy (%)');plt.tight_layout();plt.savefig(FIGURES/'new_accuracy.png',dpi=180);plt.close()
attn=pd.read_csv(ATTN_OUT/'attention_per_layer_metrics.csv')
fig,ax=plt.subplots(figsize=(9,5))
for name,group in attn.groupby('comparison'):
    layer=group.groupby('layer').cosine.mean();ax.plot(layer.index,layer.values,label=name)
ax.set(xlabel='Layer',ylabel='Mean query-averaged cosine');ax.legend();fig.tight_layout();fig.savefig(FIGURES/'real_video_attention_cosine.png',dpi=180);plt.close(fig)
fig,ax=plt.subplots(figsize=(7,5))
eff.boxplot(column='wall_s',by='method',ax=ax);ax.set(xlabel='Method',ylabel='Per-video wall time (s)');ax.set_title('')
fig.suptitle('');fig.tight_layout();fig.savefig(FIGURES/'interleaved_wall_time.png',dpi=180);plt.close(fig)
display(accuracy.round(3));display(paired_general.round(4));display(ablation_pairs.round(4));display(full_pair.round(4));display(eff_summary.round(4))

In [ ]:
# Record provenance without dumping environment variables, credentials, proxies, or tokens.
(EXPORTS/'environment_freeze.txt').write_text(run([PY312,'-m','pip','freeze'],env=model_env).stdout)
(EXPORTS/'git_status.txt').write_text(run(['git','status','--short'],cwd=ROOT).stdout)
provenance={'created_utc':datetime.now(timezone.utc).isoformat(),'reproduction_head':REPRO_HEAD,
 'official_hermes_revision':HERMES_SHA,'transformers_revision':TF_SHA,'model_snapshot':MODEL_SHA,
 'sample_fps':FPS,'seed':SEED,'gpu':smi.stdout.strip(),'python':platform.python_version(),
 'scope':{'prior_videos':50,'generalization_videos':50,'ablation_videos':50,
          'efficiency_video_runs':60,'full_videos':498,'full_questions':2495},
 'full_primary_comparison':'HERMES_4000 vs FIFO_4000',
 'native_llava_baseline':False,'near_uncompressed_50K_is_native_baseline':False,
 'optional_50K_status':[x for x in GENERAL_STATUS if x['name']=='HERMES_50000'],
 'source_verification':SOURCE_VERIFICATION,'official_tracked_source_modified':False}
(EXPORTS/'research_provenance.json').write_text(json.dumps(provenance,indent=2))
# Reconfirm tracked source was not edited by any ablation or control.
source_diff=run(['git','diff','--name-only',HERMES_SHA,'--','inference','video_qa'],cwd=ROOT,check=False)
if source_diff.returncode==0:assert not source_diff.stdout.strip(),source_diff.stdout
attn_prov=json.loads((ATTN_OUT/'attention_test_provenance.json').read_text())
full_delta=full_pair.iloc[0]
if full_delta.video_cluster_CI95_low_pp>0 or full_delta.video_cluster_CI95_high_pp<0:
    full_interpretation='The clustered interval excludes zero for the primary full-benchmark contrast; inspect direction and effect size rather than relying on a point estimate.'
else:
    full_interpretation='No statistically clear accuracy difference was detected for the primary full-benchmark contrast; this is not proof of equivalence.'
report=f'''# HERMES five-stage research report

Generated from completed, validated outputs on {datetime.now(timezone.utc).isoformat()}.

## Scope and evidence rules

- Prior samples 201–250 were reused after hash and annotation validation; none were rerun.
- New generalization and ablation runs use non-overlapping samples 251–300 at 0.5 fps, seed 2024, greedy decoding with the official repetition penalty.
- Full benchmark: 498 videos / 2,495 questions per method; primary contrast HERMES-4K versus FIFO-4K.
- The 50K configuration is a near-uncompressed HERMES control, not a native LLaVA baseline.
- Partial runs were not included in accuracy tables. Any optional 50K failure remains recorded in run status and logs.
- Exact discordance tests are exploratory; intervals resample whole videos. Repeated efficiency measurements are not independent accuracy samples.

## 1. Real-video attention

Real visual cache: {attn_prov['visual_tokens']} visual tokens, {attn_prov['past_tokens']} total past tokens, {attn_prov['layers']} layers, 16 frames. Top-k={attn_prov['topk_budget_per_layer']} is constraining.

{markdown_table(attention_summary.round(6))}

{markdown_table(selection_summary.round(6))}

These metrics test numerical and selection agreement. They do not by themselves establish that a corrected path would improve QA accuracy.

## 2. Generalization

{markdown_table(accuracy[accuracy.scope.eq('generalization_251_300')].round(4))}

{markdown_table(paired_general.round(5))}

## 3. Component ablations

{markdown_table(accuracy[accuracy.scope.eq('ablation_251_300')].round(4))}

{markdown_table(ablation_pairs.round(5))}

The uniform-layer variant uses one mixed-query/recency policy at every depth; other mechanisms remain as documented in the saved transformed methods.

## 4. Interleaved efficiency

{markdown_table(eff_summary.round(5))}

Wall time is end-to-end per-video analyzer time. Encoding, pseudo-query, pruning, and answer events are separately instrumented. TTFT from legacy logs is not used as end-to-end latency.

## 5. Full benchmark

{markdown_table(accuracy[accuracy.scope.eq('full_498')].round(4))}

{markdown_table(full_pair.round(5))}

{full_interpretation}

## Reuse and completion status

{markdown_table(pd.DataFrame(GENERAL_STATUS))}

Full shard completion is in `full_benchmark_shard_status.csv`; per-video raw predictions and timing events are included in the archive. This is a shard/benchmark-specific result, not evidence that HERMES can never help under other models, budgets, or workloads.
'''
(EXPORTS/'final_research_report.md').write_text(report)
print(report)

In [ ]:
# Final revalidation before packaging.
assert len(FULL_RESULTS['HERMES_4000'])==len(FULL_RESULTS['FIFO_4000'])==2495
assert len(pd.read_csv(EXPORTS/'efficiency_interleaved_video_metrics.csv'))==60
assert len(ABLATION_RESULTS)==4
for required in ('master_accuracy_all_stages.csv','generalization_paired.csv','ablation_paired.csv',
                 'full_benchmark_paired.csv','efficiency_interleaved_summary.csv','final_research_report.md'):
    assert (EXPORTS/required).is_file(),required
ARCHIVE=WORK/'HERMES_AutoDL_five_stage_evidence.zip'
ARCHIVE.unlink(missing_ok=True)
MANIFEST=EXPORTS/'five_stage_manifest_sha256.csv';MANIFEST.unlink(missing_ok=True)
payload=[]
def add_file(path,arcname,origin):
    path=Path(path);assert path.is_file(),path
    payload.append({'path':arcname,'origin':origin,'bytes':path.stat().st_size,'sha256':sha256(path),'source':path})
# Prior evidence stays under a distinct immutable prefix.
for path in sorted(PRESERVED.rglob('*')):
    if path.is_file():add_file(path,'preserved_final/'+path.relative_to(PRESERVED).as_posix(),'preserved')
# Only evidence-bearing new paths; never package model weights, videos, caches, environments, or downloads.
for name in ('results','logs','scripts','exports','data','efficiency_interleaved'):
    base=WORK/name
    for path in sorted(base.rglob('*')):
        if path.is_file() and path!=MANIFEST:add_file(path,path.relative_to(WORK).as_posix(),'generated')
for path in (WORK/'source_provenance.json',WORK/'download_model.py'):
    if path.is_file():add_file(path,path.name,'generated')
for candidate in (ROOT/'hermes-five-stage-autodl.ipynb',ROOT/'hermes-five-stage-autodl (1).ipynb'):
    if candidate.is_file():add_file(candidate,'notebook/'+candidate.name,'generated');break
manifest_out=pd.DataFrame([{k:v for k,v in row.items() if k!='source'} for row in payload])
assert manifest_out.path.is_unique and payload
manifest_out.to_csv(MANIFEST,index=False)
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for row in payload:archive.write(row['source'],row['path'])
    archive.write(MANIFEST,MANIFEST.relative_to(WORK).as_posix())
with zipfile.ZipFile(ARCHIVE) as archive:
    assert archive.testzip() is None
    zipped_manifest=pd.read_csv(archive.open(MANIFEST.relative_to(WORK).as_posix()))
    names=set(archive.namelist());expected=set(zipped_manifest.path)|{MANIFEST.relative_to(WORK).as_posix()}
    assert names==expected,(names-expected,expected-names)
    for row in zipped_manifest.itertuples():
        digest=hashlib.sha256(archive.read(row.path)).hexdigest()
        assert digest==row.sha256 and archive.getinfo(row.path).file_size==row.bytes,row.path
archive_sha=sha256(ARCHIVE)
print('FIVE-STAGE SUITE INTEGRITY: PASS')
print('Archive:',ARCHIVE,'\nPayload files:',len(payload),'\nArchive SHA-256:',archive_sha)